# Alarm Episode & Control-Action Coverage (16 Alarm Tags)

For each of the **16 alarm files** that have valid asset data
(every combined-event file in `DATA/combined_events/` except the two flagged tags
`03TI_1081` and `03PI_1814`, which have asset-data issues), this notebook:

1. Loads the combined events parquet file and deduplicates it.
2. Applies **trip-period filtering** (plant shutdown windows removed).
3. Extracts **PVLO / PVHI alarm episodes** (alarm-start -> OK pairs, `Category == 1`),
   using the alarm type(s) encoded in the file name.
4. Merges nearby episodes of the **same alarm type** into **alarm clusters** using a
   configurable gap (default 30 min).
5. Counts, per alarm: total clusters and clusters that have **control actions**
   (`OP` / `SP` / `MODE` changes by **any tag**) inside a configurable window
   (default: 30 min before cluster start until 30 min after cluster end).

**PVLO and PVHI are always analyzed separately**, so a file covering both conditions
produces two summary rows (one PVLO, one PVHI).

Extraction / clustering / control-action logic mirrors `SRC/alarm_episode_pipeline.ipynb`.

## 1. Configuration

In [19]:
import os
import glob

# -- Paths (fixed repo root, matches alarm_episode_pipeline) --
REPO_ROOT           = '/home/h604827/ControlActions'
DATA_DIR            = os.path.join(REPO_ROOT, 'DATA')
COMBINED_EVENTS_DIR = os.path.join(DATA_DIR, 'combined_events')
TRIP_FILE           = os.path.join(DATA_DIR, 'Final_List_Trip_Duration.csv')

# -- Tags excluded due to asset-data issues (notified to the other team) --
EXCLUDE_TAGS = {'03TI_1081', '03PI_1814'}

# -- Alarm extraction --
ALARM_CATEGORY   = 1                  # Category value that marks real alarms
ALARM_CONDITIONS = ('PVLO', 'PVHI')   # condition types considered (taken from each file name)

# -- Clustering: merge nearby alarm episodes into one alarm cluster --
CLUSTER_GAP_MINUTES = 30              # a gap > this starts a new cluster

# -- Control-action window (the configurable 'half hour') --
ACTION_WINDOW_MINUTES       = 30                     # minutes before cluster start AND after cluster end
CONTROL_ACTION_DESCRIPTIONS = ['OP', 'SP', 'MODE']   # CHANGE descriptions counted as control actions
RESTRICT_ACTIONS_TO_TARGET_TAG = False               # False = any tag's OP/SP/MODE counts (matches pipeline)

# -- PVLO and PVHI are always analyzed separately (one summary row per alarm tag / type). --
CLUSTER_CONDITIONS_TOGETHER = False

# -- Orphan handling for alarm episodes (see extract_episodes for full rules):
#    A trailing alarm start that never clears (no 'OK' before data ends) is an 'unclosed' alarm.
#    True  -> count it as a (zero-length) episode;  False -> drop it (matches reference pipeline). --
COUNT_UNCLOSED_STARTS = False

# -- Event start-date cutoff. '2022-01-01' matches alarm_episode_pipeline (PV/OP data starts
#    2022-01-03) and reproduces the reference cluster counts (e.g. 1071 PVLO -> 539 clusters).
#    Set to None to use the full event history. Trip filtering is applied separately. --
START_DATE = '2022-01-01'

# -- Trip filtering --
FILTER_TRIPS = True

# -- Save summary to RESULTS/ --
SAVE_RESULTS = True
RESULTS_DIR  = os.path.join(REPO_ROOT, 'RESULTS', 'alarm_control_action_coverage')

print('Combined events dir       :', COMBINED_EVENTS_DIR)
print('Trip file                 :', TRIP_FILE)
print('Excluded tags             :', EXCLUDE_TAGS)
print('Cluster gap               :', CLUSTER_GAP_MINUTES, 'min')
print('Action window             : +/-', ACTION_WINDOW_MINUTES, 'min around each cluster')
print('Control-action types       :', CONTROL_ACTION_DESCRIPTIONS)
print('Cluster PVLO+PVHI together :', CLUSTER_CONDITIONS_TOGETHER)
print('Count unclosed starts     :', COUNT_UNCLOSED_STARTS)
print('Start-date cutoff         :', START_DATE)

Combined events dir       : /home/h604827/ControlActions/DATA/combined_events
Trip file                 : /home/h604827/ControlActions/DATA/Final_List_Trip_Duration.csv
Excluded tags             : {'03PI_1814', '03TI_1081'}
Cluster gap               : 30 min
Action window             : +/- 30 min around each cluster
Control-action types       : ['OP', 'SP', 'MODE']
Cluster PVLO+PVHI together : False
Count unclosed starts     : False
Start-date cutoff         : 2022-01-01


## 2. Imports

In [2]:
import numpy as np
import pandas as pd

print('pandas', pd.__version__)

pandas 2.3.3


## 3. Helper Functions

In [20]:
NEEDED_COLS = ['VT_Start', 'Source', 'ConditionName', 'Action', 'Category', 'Description']


def norm_tag(s):
    # Normalize a tag/source for matching: drop underscores, uppercase.
    return str(s).replace('_', '').upper()


def parse_alarm_file(path):
    # '..._03LIC_1016_PVLO_PVHI_combined_events.parquet' -> ('03LIC_1016', ['PVLO', 'PVHI'])
    base = os.path.basename(path).replace('_combined_events.parquet', '')
    toks = base.split('_')
    types = []
    while toks and toks[-1] in ('PVLO', 'PVHI'):
        types.insert(0, toks.pop())
    return '_'.join(toks), types


def strip_timezone(s):
    if getattr(s.dt, 'tz', None) is not None:
        return s.dt.tz_localize(None)
    return s


def load_events(path):
    # Read needed columns, parse timestamps, dedup overlapping _E/_EA/_EAL rows, sort.
    df = pd.read_parquet(path, columns=NEEDED_COLS)
    df['VT_Start'] = strip_timezone(pd.to_datetime(df['VT_Start'], errors='coerce'))
    df = df.dropna(subset=['VT_Start'])
    df = df.sort_values('VT_Start')
    df = df.drop_duplicates(subset=['VT_Start', 'Source', 'ConditionName', 'Description'],
                            keep='first').reset_index(drop=True)
    if START_DATE is not None:
        df = df[df['VT_Start'] >= pd.to_datetime(START_DATE)].reset_index(drop=True)
    return df


def apply_trip_filter(events_df, trips_df):
    # Remove events that fall inside any trip (plant-shutdown) window [Stop Date, Start Date].
    if not FILTER_TRIPS or trips_df is None or trips_df.empty:
        return events_df
    vt = events_df['VT_Start']
    mask = pd.Series(False, index=events_df.index)
    for _, trip in trips_df.iterrows():
        mask |= (vt >= trip['Stop Date']) & (vt <= trip['Start Date'])
    return events_df[~mask].reset_index(drop=True)


def extract_episodes(events_df, tag, condition):
    # Pair each alarm start (Action blank) with the next alarm clear (Action == 'OK')
    # for one condition, walking events in time order.
    #
    # Orphan handling (explicit):
    #   * start while an alarm is already open  -> re-trigger of the SAME still-active alarm;
    #     it does NOT open a new episode (counted as orphan_start).
    #   * 'OK' while no alarm is open           -> clears an alarm whose start was not captured
    #     (missing / out-of-window start); skipped (counted as orphan_end).
    #   * a start still open when events run out -> never cleared (counted as unclosed_start);
    #     added as a zero-length episode only if COUNT_UNCLOSED_STARTS is True.
    #
    # Returns (episodes_df, stats) where stats has orphan_starts / orphan_ends / unclosed_starts.
    ns = events_df['Source'].map(norm_tag)
    sub = events_df[(ns == norm_tag(tag)) &
                    (events_df['ConditionName'] == condition) &
                    (events_df['Category'] == ALARM_CATEGORY)].sort_values('VT_Start')

    episodes = []
    current_start = None
    orphan_starts = 0
    orphan_ends = 0
    unclosed_starts = 0

    for _, row in sub.iterrows():
        action = row['Action']
        is_start = pd.isna(action) or str(action).strip() == ''
        is_end = (action == 'OK')

        if is_start:
            if current_start is None:
                current_start = row['VT_Start']
            else:
                orphan_starts += 1                      # re-trigger while still open
        elif is_end:
            if current_start is not None:
                episodes.append({'alarm_start': current_start,
                                 'alarm_end': row['VT_Start'],
                                 'condition': condition})
                current_start = None
            else:
                orphan_ends += 1                        # OK with no open start

    if current_start is not None:
        unclosed_starts += 1                            # trailing start that never cleared
        if COUNT_UNCLOSED_STARTS:
            episodes.append({'alarm_start': current_start,
                             'alarm_end': current_start,  # unknown clear -> zero-length episode
                             'condition': condition})

    stats = {'orphan_starts': orphan_starts,
             'orphan_ends': orphan_ends,
             'unclosed_starts': unclosed_starts}
    return pd.DataFrame(episodes), stats


def cluster_episodes(episodes, gap_minutes):
    # Merge episodes whose inter-episode gap <= gap_minutes into one alarm cluster.
    if episodes is None or episodes.empty:
        return pd.DataFrame(columns=['cluster_id', 'cluster_start', 'cluster_end', 'n_alarms'])
    ep = episodes.sort_values('alarm_start').reset_index(drop=True)
    gap_to_next = (ep['alarm_start'].shift(-1) - ep['alarm_end']).dt.total_seconds() / 60.0
    cluster_ids = [0]
    cid = 0
    for g in gap_to_next.iloc[:-1]:
        if pd.notna(g) and g <= gap_minutes:
            cluster_ids.append(cid)
        else:
            cid += 1
            cluster_ids.append(cid)
    ep['cluster_id'] = cluster_ids
    clusters = ep.groupby('cluster_id').agg(
        cluster_start=('alarm_start', 'min'),
        cluster_end=('alarm_end', 'max'),
        n_alarms=('alarm_start', 'count')).reset_index()
    return clusters


def get_action_times(events_df, tag):
    # Sorted numpy array of timestamps for OP/SP/MODE CHANGE events (control actions).
    chg = events_df[(events_df['ConditionName'] == 'CHANGE') &
                    (events_df['Description'].isin(CONTROL_ACTION_DESCRIPTIONS))]
    if RESTRICT_ACTIONS_TO_TARGET_TAG:
        ns = chg['Source'].map(norm_tag)
        chg = chg[ns == norm_tag(tag)]
    return np.sort(chg['VT_Start'].values.astype('datetime64[ns]'))


def flag_clusters_with_actions(clusters, action_times, window_minutes):
    # Mark each cluster True if any control action falls in
    # [cluster_start - window, cluster_end + window].
    if clusters is None or clusters.empty:
        return []
    if action_times.size == 0:
        return [False] * len(clusters)
    w = pd.Timedelta(minutes=window_minutes)
    flags = []
    for _, c in clusters.iterrows():
        lo = np.searchsorted(action_times, np.datetime64(c['cluster_start'] - w), side='left')
        hi = np.searchsorted(action_times, np.datetime64(c['cluster_end'] + w), side='right')
        flags.append(bool(hi > lo))
    return flags


print('Helper functions defined.')

Helper functions defined.


In [4]:
# -- Load trip periods once --
trips_df = None
if FILTER_TRIPS:
    trips_df = pd.read_csv(TRIP_FILE)
    trips_df['Stop Date'] = pd.to_datetime(trips_df['Stop Date'])
    trips_df['Start Date'] = pd.to_datetime(trips_df['Start Date'])
    print('Loaded', len(trips_df), 'trip periods from', os.path.basename(TRIP_FILE))
else:
    print('Trip filtering disabled.')

Loaded 106 trip periods from Final_List_Trip_Duration.csv


## 4. Process All 16 Alarm Tags

Discovers every combined-event file, skips the two flagged tags, then for each file
extracts + clusters alarms and counts clusters that have control actions.

In [5]:
# -- Discover files to process --
all_files = sorted(glob.glob(os.path.join(COMBINED_EVENTS_DIR, '*_combined_events.parquet')))
work = []
for f in all_files:
    tag, types = parse_alarm_file(f)
    if tag in EXCLUDE_TAGS:
        print('Skip (flagged asset data):', os.path.basename(f))
        continue
    types = [t for t in types if t in ALARM_CONDITIONS]
    if not types:
        print('Skip (no PVLO/PVHI in name):', os.path.basename(f))
        continue
    work.append((tag, types, f))

print()
print('Files to process:', len(work))

Skip (flagged asset data): 03TI_1081_PVHI_combined_events.parquet

Files to process: 16


In [25]:
summary_rows  = []
cluster_store = {}   # key 'tag TYPE' -> clusters DataFrame (with has_control_actions flag)

for tag, types, path in work:
    print('Processing', tag, types, '...')
    events_df    = load_events(path)
    events_df    = apply_trip_filter(events_df, trips_df)
    action_times = get_action_times(events_df, tag)

    # Decide how PVLO / PVHI are grouped into summary rows
    if CLUSTER_CONDITIONS_TOGETHER:
        groups = {'+'.join(types): types}
    else:
        groups = {t: [t] for t in types}

    for label, conds in groups.items():
        ep_parts  = []
        grp_stats = {'orphan_starts': 0, 'orphan_ends': 0, 'unclosed_starts': 0}
        for c in conds:
            ep_c, st_c = extract_episodes(events_df, tag, c)
            if ep_c is not None and not ep_c.empty:
                ep_parts.append(ep_c)
            for k in grp_stats:
                grp_stats[k] += st_c[k]

        if ep_parts:
            episodes = pd.concat(ep_parts, ignore_index=True).sort_values('alarm_start').reset_index(drop=True)
        else:
            episodes = pd.DataFrame(columns=['alarm_start', 'alarm_end', 'condition'])

        clusters = cluster_episodes(episodes, CLUSTER_GAP_MINUTES)
        flags    = flag_clusters_with_actions(clusters, action_times, ACTION_WINDOW_MINUTES)
        if not clusters.empty:
            clusters = clusters.copy()
            clusters['has_control_actions'] = flags

        n_clusters = len(clusters)
        n_act      = int(np.sum(flags)) if flags else 0
        key        = tag + ' ' + label
        cluster_store[key] = clusters

        summary_rows.append({
            'alarm_tag': tag,
            'alarm_type': label,
            'raw_alarm_episodes': len(episodes),
            'merged_alarm_episodes': n_clusters,
            'episodes_with_control_actions': n_act,
            'pct_with_control_actions': round(100.0 * n_act / n_clusters, 1) if n_clusters else 0.0,
            'orphan_starts': grp_stats['orphan_starts'],
            'orphan_ends': grp_stats['orphan_ends'],
            'unclosed_starts': grp_stats['unclosed_starts'],
            'first_alarm': episodes['alarm_start'].min() if len(episodes) else pd.NaT,
            'last_alarm': episodes['alarm_start'].max() if len(episodes) else pd.NaT,
            'events_file': os.path.basename(path),
        })
        print(f'    {key}: raw_episodes={len(episodes)}  clusters={n_clusters}  with_actions={n_act}'
              f'  (orphan_starts={grp_stats["orphan_starts"]}, orphan_ends={grp_stats["orphan_ends"]},'
              f' unclosed={grp_stats["unclosed_starts"]})')

summary_df = pd.DataFrame(summary_rows).sort_values(['alarm_tag', 'alarm_type']).reset_index(drop=True)
print()
print('Done. Summary rows:', len(summary_df))

Processing 03FIC_1668 ['PVHI'] ...
    03FIC_1668 PVHI: raw_episodes=6  clusters=3  with_actions=2  (orphan_starts=0, orphan_ends=0, unclosed=0)
Processing 03LIC1608 ['PVLO'] ...
    03LIC1608 PVLO: raw_episodes=16  clusters=11  with_actions=11  (orphan_starts=0, orphan_ends=0, unclosed=0)
Processing 03LIC_1016 ['PVLO', 'PVHI'] ...
    03LIC_1016 PVLO: raw_episodes=1959  clusters=594  with_actions=368  (orphan_starts=5, orphan_ends=3, unclosed=0)
    03LIC_1016 PVHI: raw_episodes=141  clusters=63  with_actions=56  (orphan_starts=0, orphan_ends=0, unclosed=0)
Processing 03LIC_1071 ['PVLO', 'PVHI'] ...
    03LIC_1071 PVLO: raw_episodes=1379  clusters=539  with_actions=458  (orphan_starts=2, orphan_ends=10, unclosed=0)
    03LIC_1071 PVHI: raw_episodes=182  clusters=38  with_actions=35  (orphan_starts=1, orphan_ends=1, unclosed=0)
Processing 03LIC_1619 ['PVLO', 'PVHI'] ...
    03LIC_1619 PVLO: raw_episodes=657  clusters=276  with_actions=204  (orphan_starts=2, orphan_ends=4, unclosed=0)
 

## 5. Summary Table

In [27]:
from IPython.display import display

show_cols = ['alarm_tag', 'alarm_type', 'raw_alarm_episodes', 'merged_alarm_episodes',
             'episodes_with_control_actions', 'pct_with_control_actions',
             'orphan_starts', 'orphan_ends', 'unclosed_starts',
             'first_alarm', 'last_alarm']
display(summary_df[show_cols])

print()
print('Totals across', len(summary_df), 'alarms:')
print('  raw alarm episodes            :', int(summary_df['raw_alarm_episodes'].sum()))
print('  alarm clusters (merged)       :', int(summary_df['merged_alarm_episodes'].sum()))
print('  clusters with control actions :', int(summary_df['episodes_with_control_actions'].sum()))
print('  orphan starts (re-triggers)   :', int(summary_df['orphan_starts'].sum()))
print('  orphan ends   (OK, no start)  :', int(summary_df['orphan_ends'].sum()))
print('  unclosed starts (never OK)    :', int(summary_df['unclosed_starts'].sum()))

,alarm_tag,alarm_type,raw_alarm_episodes,merged_alarm_episodes,episodes_with_control_actions,pct_with_control_actions,orphan_starts,orphan_ends,unclosed_starts,first_alarm,last_alarm
0,03FIC_1668,PVHI,6,3,2,66.7,0,0,0,2024-01-06 10:44:02.103500,2024-12-26 09:42:43.596300
1,03LIC1608,PVLO,16,11,11,100.0,0,0,0,2022-06-12 02:05:53.852700,2025-05-23 08:50:32.577900
2,03LIC_1016,PVHI,141,63,56,88.9,0,0,0,2022-04-09 22:19:19.003700,2025-06-21 18:27:26.523600
3,03LIC_1016,PVLO,1959,594,368,62.0,5,3,0,2022-01-04 03:44:17.703100,2025-06-27 17:31:42.385000
4,03LIC_1071,PVHI,182,38,35,92.1,1,1,0,2022-10-30 18:48:28.102900,2025-05-09 10:15:47.430600
5,03LIC_1071,PVLO,1379,539,458,85.0,2,10,0,2022-01-05 08:53:41.852900,2025-06-22 17:14:12.209300
6,03LIC_1619,PVHI,773,357,255,71.4,0,0,0,2022-02-28 01:20:23.503700,2025-06-17 18:29:03.791200
7,03LIC_1619,PVLO,657,276,204,73.9,2,4,0,2022-04-19 02:57:44.351800,2025-06-16 20:11:11.509500
8,03PIC1023,PVHI,332,113,100,88.5,0,0,0,2022-06-05 14:12:32.552600,2025-06-17 17:10:33.208200
9,03PIC1023,PVLO,13,7,7,100.0,2,0,0,2022-01-23 04:38:25.502200,2024-05-01 23:53:23.303300



Totals across 21 alarms:
  raw alarm episodes            : 7622
  alarm clusters (merged)       : 3838
  clusters with control actions : 2878
  orphan starts (re-triggers)   : 24
  orphan ends   (OK, no start)  : 32
  unclosed starts (never OK)    : 1


In [28]:
summary_df[summary_df['alarm_tag'].isin(['03LIC_1071', '03LIC_1016', '03TIC_1023', '03PIC_1104', '03LIC_1619', '03FIC_1668', '03TIC_1009'])][['alarm_tag', 'alarm_type', 'merged_alarm_episodes', 'episodes_with_control_actions']]

,alarm_tag,alarm_type,merged_alarm_episodes,episodes_with_control_actions
0,03FIC_1668,PVHI,3,2
2,03LIC_1016,PVHI,63,56
3,03LIC_1016,PVLO,594,368
4,03LIC_1071,PVHI,38,35
5,03LIC_1071,PVLO,539,458
6,03LIC_1619,PVHI,357,255
7,03LIC_1619,PVLO,276,204
11,03PIC_1104,PVHI,133,129
13,03TIC_1009,PVHI,3,3
14,03TIC_1009,PVLO,40,30


In [23]:
# -- Save the summary --
if SAVE_RESULTS:
    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_xlsx = os.path.join(RESULTS_DIR, 'alarm_control_action_coverage_summary.xlsx')
    summary_df.to_excel(out_xlsx, index=False)
    print('Saved:', out_xlsx)
else:
    print('SAVE_RESULTS = False (nothing written).')

Saved: /home/h604827/ControlActions/RESULTS/alarm_control_action_coverage/alarm_control_action_coverage_summary.xlsx


## 6. Inspect a Single Alarm (optional)

Pick any key from `cluster_store` to see its per-cluster detail, including which
clusters had control actions.

In [9]:
print('Available keys:')
for k in cluster_store:
    print('  ', k)

inspect_key = list(cluster_store)[0]
print()
print('Showing:', inspect_key)
display(cluster_store[inspect_key].head(20))

Available keys:
   03FIC_1668 PVHI
   03LIC1608 PVLO
   03LIC_1016 PVLO+PVHI
   03LIC_1071 PVLO+PVHI
   03LIC_1619 PVLO+PVHI
   03PIC1023 PVLO+PVHI
   03PIC_1013 PVLO
   03PIC_1104 PVHI
   03PI_1655 PVHI
   03TIC_1009 PVHI
   03TIC_1009 PVLO
   03TIC_1023 PVLO+PVHI
   03TIC_1145 PVLO
   03TIC_1635 PVHI
   03TIC_1635 PVLO
   03TIC_1745A PVHI

Showing: 03FIC_1668 PVHI


,cluster_id,cluster_start,cluster_end,n_alarms,has_control_actions
0,0,2021-12-07 02:28:58.902700,2021-12-07 02:29:05.903600,1,True
1,1,2024-01-06 10:44:02.103500,2024-01-06 10:44:40.102600,1,True
2,2,2024-04-28 08:45:14.752000,2024-04-28 08:47:11.751500,2,True
3,3,2024-12-26 09:41:06.205600,2024-12-26 09:42:48.471500,3,False
